<a href="https://colab.research.google.com/github/aferenaz/MLOPSII-tp6/blob/main/mini_tp6_Ferenaz_Andrea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-TP 6 · Tu modelo en el Data Lake (individual)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

## Consigna
Sube el artefacto de **tu propio modelo** a un Data Lake (MinIO/S3) y **sírvelo cargándolo
desde allí**, no desde disco. Organiza el bucket con **zonas** y **versión**.

### Qué se entrega
1. El bucket con zonas (`raw`, `curated`, `models`) y tu modelo en `models/v1/…`.
2. Una función que **carga el modelo desde el lake** y predice.
3. (Opcional) El experimento registrado con **MLflow apuntando a MinIO/S3**.
4. Una reflexión breve: ¿por qué servir desde el lake y no hornear el modelo en la imagen?

### Se evalúa
- Que **corra de punta a punta**.
- Uso correcto de la API S3 (`boto3`): bucket, zonas por prefijo, versión del modelo.
- Que el modelo se **cargue desde el lake** para servir.
- (Opcional) Integración con MLflow.

> Antes de empezar, levanta MinIO con Docker (ver la *Guía de la práctica* o el tutorial
> `Teoria/datalake_tutorial.ipynb`) y instala las dependencias:
> `uv add boto3 pandas pyarrow scikit-learn mlflow joblib`

---

**Modelo:** el de mi Sesión 1 (Mini-TP 1): forecasting de demanda diaria para **Gout** (cadena
sin TACC, 9 sucursales, 96 SKUs), `RandomForestRegressor` sobre 11 features, `gout-demanda-rf`.

**Cómo correrlo**
- **Local:** levantar MinIO con Docker como indica el tutorial y ejecutar todo.
- **Google Colab:** no hay Docker. Si la celda *0.b* no encuentra un MinIO escuchando en
  `localhost:9000`, levanta un servidor compatible con S3 (moto) instalado con `pip`. En local, con
  el contenedor de MinIO corriendo, no hace nada.


## 0. Configuración (dada)
Credenciales por variables de entorno y endpoint de MinIO. No modifiques salvo que tu
MinIO use otro puerto/credenciales.


In [ ]:
# Dependencias (equivalente uv: uv add boto3 pandas pyarrow scikit-learn mlflow joblib)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "boto3", "pandas", "pyarrow", "scikit-learn", "mlflow", "joblib"])
print("Listo.")


Listo.


In [ ]:
import os
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"
BUCKET = "datalake-tp6"      # usa tu propio bucket
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET)

Endpoint: http://localhost:9000 | Bucket: datalake-tp6


### 0.b S3 sin Docker (sólo si hace falta, pensado para Colab)

Si ya hay un MinIO respondiendo en `ENDPOINT` (por ejemplo, el contenedor de Docker del
tutorial), esta celda no hace nada.

Si no, levanta en segundo plano un servidor **compatible con la API de S3** que se instala con
`pip` ([moto](https://github.com/getmoto/moto)). MinIO dejó de publicar binarios precompilados
de su edición comunitaria (sólo distribuye el código fuente) y en Colab no hay Docker. Para el
código del TP no cambia nada: `boto3` habla el mismo protocolo, con el mismo endpoint y las
mismas credenciales. La diferencia es que moto guarda todo en memoria: si se reinicia el entorno
de ejecución, el lake se vacía y hay que volver a correr el notebook.

In [ ]:
import os, sys, time, subprocess, urllib.request, urllib.error


def s3_vivo(endpoint=ENDPOINT):
    """True si algo responde HTTP en el endpoint (MinIO o cualquier S3 compatible)."""
    try:
        urllib.request.urlopen(endpoint, timeout=2)
        return True
    except urllib.error.HTTPError:      # 403/400: el servidor está, sólo pide autenticación
        return True
    except Exception:
        return False


if s3_vivo():
    print(f"Ya hay un servidor S3 escuchando en {ENDPOINT}: lo uso.")
else:
    print("No hay MinIO en el endpoint: levanto un servidor S3 compatible (moto) en segundo plano...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moto[server]"], check=True)
    subprocess.Popen([sys.executable, "-m", "moto.server", "-H", "127.0.0.1", "-p", "9000"],
                     stdout=open("/tmp/s3.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if s3_vivo():
            break
        time.sleep(1)
    print(f"Servidor S3 listo en {ENDPOINT}." if s3_vivo()
          else "El servidor S3 no respondió: revisá /tmp/s3.log")


No hay MinIO en el endpoint: levanto un servidor S3 compatible (moto) en segundo plano...
Servidor S3 listo en http://localhost:9000.


In [ ]:
import boto3
from botocore.client import Config

# Un único cliente S3: el mismo código sirve para MinIO y para S3 real (sin endpoint_url).
# Las credenciales salen de las variables de entorno, no del código.
s3 = boto3.client("s3", endpoint_url=ENDPOINT, config=Config(signature_version="s3v4"))
print("Cliente S3 listo. Buckets existentes:", [b["Name"] for b in s3.list_buckets()["Buckets"]])


Cliente S3 listo. Buckets existentes: []


## 1. Crear el bucket y las zonas


In [ ]:
def crear_bucket(bucket):
    existentes = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if bucket in existentes:
        print(f"El bucket '{bucket}' ya existe.")
    else:
        s3.create_bucket(Bucket=bucket)
        print(f"Bucket '{bucket}' creado.")


crear_bucket(BUCKET)

# En S3 no hay carpetas: las zonas son prefijos y se materializan con el primer objeto.
ZONAS = ["raw", "curated", "models"]
for z in ZONAS:
    s3.put_object(Bucket=BUCKET, Key=f"{z}/.keep", Body=b"")
print("Zonas inicializadas:", ZONAS)


Bucket 'datalake-tp6' creado.
Zonas inicializadas: ['raw', 'curated', 'models']


## 2. Entrenar tu modelo

Uso **mi modelo** del curso: el de demanda de Gout del Mini-TP 1, con la misma receta de
`train.py` (datos sintéticos del dominio, misma semilla y mismos hiperparámetros).

El dataset sigue el recorrido del lake: el crudo va a `raw` tal como llega (CSV, con nombres y
tipos de origen), y la versión curada (Parquet, con las features tipadas y listas para entrenar)
va a `curated`. El modelo se entrena leyendo el dataset **desde `curated` en el lake**, no desde
la memoria, para que el artefacto quede atado a un dataset concreto y reproducible.


In [ ]:
import joblib, io, json, hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

FEATURES = [
    "sucursal_id", "sku_id", "dia_semana", "mes", "es_finde", "es_feriado",
    "en_promocion", "precio_unitario", "demanda_lag_1", "demanda_lag_7", "media_movil_7",
]
TARGET = "demanda"
DATASET = "gout_demanda"


def datos_mini_tp1(n=8_000, seed=42):
    """Datos sintéticos de Gout, idénticos a los de train.py del Mini-TP 1."""
    rng = np.random.default_rng(seed)
    sucursal_id = rng.integers(1, 10, n)
    sku_id = rng.integers(1, 97, n)
    dia_semana = rng.integers(0, 7, n)
    mes = rng.integers(1, 13, n)
    es_finde = (dia_semana >= 5).astype(int)
    es_feriado = (rng.random(n) < 0.05).astype(int)
    en_promocion = (rng.random(n) < 0.20).astype(int)
    precio_unitario = rng.uniform(800, 3500, n)
    demanda_lag_1 = rng.uniform(0, 120, n)
    demanda_lag_7 = rng.uniform(0, 120, n)
    media_movil_7 = np.clip((demanda_lag_1 + demanda_lag_7) / 2 + rng.normal(0, 8, n), 0, None)
    base = (18 + 0.35 * demanda_lag_1 + 0.30 * demanda_lag_7 + 0.25 * media_movil_7
            + 14 * es_finde + 20 * es_feriado + 12 * en_promocion - 0.004 * precio_unitario
            + 4 * np.sin(2 * np.pi * mes / 12) + (sku_id % 7))
    y = np.clip(base + rng.normal(0, 6, n), 0, None)
    X = np.column_stack([sucursal_id, sku_id, dia_semana, mes, es_finde, es_feriado,
                         en_promocion, precio_unitario, demanda_lag_1, demanda_lag_7, media_movil_7])
    return pd.DataFrame(X, columns=FEATURES).assign(**{TARGET: y})


# Dataset crudo "como llega del POS": nombres y formatos de origen, sin tipar.
df = datos_mini_tp1()
df_raw = df.rename(columns={"sucursal_id": "Sucursal", "sku_id": "SKU", "precio_unitario": "Precio ($)",
                            TARGET: "Unidades vendidas"})
df_raw["Precio ($)"] = df_raw["Precio ($)"].map(lambda p: f"{p:,.2f}")     # texto, como en el export
print("Dataset crudo:", df_raw.shape)
df_raw.head(3)


Dataset crudo: (8000, 12)


,Sucursal,SKU,dia_semana,mes,es_finde,es_feriado,en_promocion,Precio ($),demanda_lag_1,demanda_lag_7,media_movil_7,Unidades vendidas
0,1.0,63.0,6.0,12.0,1.0,0.0,0.0,"1,207.31",34.970371,94.617406,68.069701,85.251256
1,7.0,69.0,3.0,7.0,0.0,0.0,0.0,"2,066.40",14.500379,69.067450,56.569598,58.318922
2,6.0,26.0,0.0,8.0,0.0,0.0,1.0,"1,668.04",12.884746,43.898570,25.400214,55.479065


In [ ]:
def curar(df_raw: pd.DataFrame) -> pd.DataFrame:
    """raw -> curated: nombres canónicos, tipos correctos, validaciones mínimas."""
    d = df_raw.rename(columns={"Sucursal": "sucursal_id", "SKU": "sku_id",
                               "Precio ($)": "precio_unitario", "Unidades vendidas": TARGET})
    d["precio_unitario"] = d["precio_unitario"].str.replace(",", "", regex=False).astype(float)
    enteras = ["sucursal_id", "sku_id", "dia_semana", "mes", "es_finde", "es_feriado", "en_promocion"]
    d[enteras] = d[enteras].astype("int16")
    assert d["sucursal_id"].between(1, 9).all() and d["sku_id"].between(1, 96).all()
    assert (d[TARGET] >= 0).all() and d[FEATURES].notna().all().all()
    return d[FEATURES + [TARGET]]


df_curado = curar(df_raw)
print("Dataset curado:", df_curado.shape)
print(df_curado.dtypes.value_counts().to_dict())


Dataset curado: (8000, 12)
{dtype('int16'): 7, dtype('float64'): 5}


## 3. Subir dataset y modelo al lake

Convención de rutas:

```
s3://datalake-tp6/
├── raw/gout_demanda/fecha=AAAA-MM-DD/ventas.csv        crudo, inmutable
├── curated/gout_demanda/fecha=AAAA-MM-DD/ventas.parquet curado, listo para entrenar
├── curated/preds/gout-demanda-rf/v1/predicciones.parquet  (sección 4)
└── models/
    ├── v1/gout-demanda-rf.pkl + metadata.json          una carpeta por versión, inmutable
    ├── v2/...
    └── production.json                                 puntero a la versión que se sirve
```

Las versiones de `models/` **no se sobrescriben nunca**: subir una versión que ya existe es un
error. Qué versión está en producción lo decide `models/production.json`, no la ruta del
servicio.


In [ ]:
HOY = datetime.now(timezone.utc).date().isoformat()
KEY_RAW = f"raw/{DATASET}/fecha={HOY}/ventas.csv"
KEY_CURADO = f"curated/{DATASET}/fecha={HOY}/ventas.parquet"
NOMBRE_MODELO = "gout-demanda-rf"

# 4a. crudo -> raw (CSV tal cual)
s3.put_object(Bucket=BUCKET, Key=KEY_RAW, Body=df_raw.to_csv(index=False).encode("utf-8"))

# 4b. curado -> curated (Parquet: columnar, tipado, ~5x más chico)
s3.put_object(Bucket=BUCKET, Key=KEY_CURADO, Body=df_curado.to_parquet(index=False))
print("Subidos:", KEY_RAW, "|", KEY_CURADO)


# 4c. entrenar LEYENDO desde curated en el lake
def leer_parquet(bucket, key):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key=key)["Body"].read()))


def entrenar(datos, **hiper):
    X, y = datos[FEATURES], datos[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = RandomForestRegressor(random_state=42, n_jobs=-1, **hiper).fit(X_tr, y_tr)
    pred = m.predict(X_te)
    return m, {"mae": round(float(mean_absolute_error(y_te, pred)), 3),
               "r2": round(float(r2_score(y_te, pred)), 4)}, X_te


datos = leer_parquet(BUCKET, KEY_CURADO)
HIPER_V1 = {"n_estimators": 120, "max_depth": 10, "min_samples_leaf": 10}   # los del Mini-TP 1
modelo, metricas_v1, X_test = entrenar(datos, **HIPER_V1)
print(f"Modelo entrenado desde s3://{BUCKET}/{KEY_CURADO} | métricas test: {metricas_v1}")


# 4d. modelo -> models/v1/ (joblib -> bytes -> put_object), con metadata y checksum
def subir_modelo(modelo, version, metricas, hiper, dataset_key):
    key_pkl = f"models/{version}/{NOMBRE_MODELO}.pkl"
    try:                                    # las versiones son inmutables
        s3.head_object(Bucket=BUCKET, Key=key_pkl)
        raise FileExistsError(f"s3://{BUCKET}/{key_pkl} ya existe: usá una versión nueva")
    except s3.exceptions.ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey"):
            raise
    buf = io.BytesIO(); joblib.dump(modelo, buf); cuerpo = buf.getvalue()
    meta = {"nombre": NOMBRE_MODELO, "version": version, "features": FEATURES,
            "hiperparametros": hiper, "metricas": metricas, "dataset": f"s3://{BUCKET}/{dataset_key}",
            "sha256": hashlib.sha256(cuerpo).hexdigest(), "bytes": len(cuerpo),
            "entrenado": datetime.now(timezone.utc).isoformat()}
    s3.put_object(Bucket=BUCKET, Key=key_pkl, Body=cuerpo)
    s3.put_object(Bucket=BUCKET, Key=f"models/{version}/metadata.json",
                  Body=json.dumps(meta, indent=2).encode("utf-8"), ContentType="application/json")
    return key_pkl, meta


def promover(version):
    """Mueve el puntero de producción. Rollback = promover la versión anterior."""
    s3.put_object(Bucket=BUCKET, Key="models/production.json",
                  Body=json.dumps({"version": version,
                                   "promovido": datetime.now(timezone.utc).isoformat()}).encode("utf-8"))
    print(f"models/production.json -> {version}")


# Si se re-ejecuta el notebook, v1 ya existe: se reutiliza en vez de pisarla.
try:
    KEY_MODELO_V1, meta_v1 = subir_modelo(modelo, "v1", metricas_v1, HIPER_V1, KEY_CURADO)
    print(f"Modelo subido: {KEY_MODELO_V1} ({meta_v1['bytes']:,} bytes, sha256 {meta_v1['sha256'][:12]}…)")
except FileExistsError as e:
    KEY_MODELO_V1 = f"models/v1/{NOMBRE_MODELO}.pkl"
    print("Aviso:", e)
promover("v1")


def listar(bucket, prefix=""):
    for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get("Contents", []):
        if not obj["Key"].endswith("/.keep"):
            print(f"  {obj['Key']:62s} {obj['Size']:>10,} bytes")


print(f"\nContenido de s3://{BUCKET}/")
listar(BUCKET)


Subidos: raw/gout_demanda/fecha=2026-10-05/ventas.csv | curated/gout_demanda/fecha=2026-10-05/ventas.parquet
Modelo entrenado desde s3://datalake-tp6/curated/gout_demanda/fecha=2026-10-05/ventas.parquet | métricas test: {'mae': 6.261, 'r2': 0.8933}
Modelo subido: models/v1/gout-demanda-rf.pkl (4,144,033 bytes, sha256 4838e4181744…)
models/production.json -> v1

Contenido de s3://datalake-tp6/
  curated/gout_demanda/fecha=2026-10-05/ventas.parquet              390,474 bytes
  models/production.json                                                 66 bytes
  models/v1/gout-demanda-rf.pkl                                   4,144,033 bytes
  models/v1/metadata.json                                               669 bytes
  raw/gout_demanda/fecha=2026-10-05/ventas.csv                      901,610 bytes


## 4. Servir el modelo desde el lake
El corazón del Mini-TP: **cargar el modelo desde el lake** (no desde disco) y predecir.


In [ ]:
def cargar_modelo_desde_lake(bucket, key):
    """Descarga el artefacto del lake y lo reconstruye en memoria (nunca toca el disco)."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return joblib.load(io.BytesIO(obj["Body"].read()))


def cargar_modelo_en_produccion(bucket=BUCKET):
    """Lo que hace el servicio al arrancar: puntero -> metadata -> artefacto verificado."""
    version = json.loads(s3.get_object(Bucket=bucket, Key="models/production.json")["Body"].read())["version"]
    meta = json.loads(s3.get_object(Bucket=bucket, Key=f"models/{version}/metadata.json")["Body"].read())
    cuerpo = s3.get_object(Bucket=bucket, Key=f"models/{version}/{NOMBRE_MODELO}.pkl")["Body"].read()
    if hashlib.sha256(cuerpo).hexdigest() != meta["sha256"]:
        raise ValueError(f"El artefacto de {version} no coincide con su checksum")
    return joblib.load(io.BytesIO(cuerpo)), meta


# Liberamos el modelo de la memoria para demostrar que lo que predice viene del lake.
del modelo

# a) la función pedida: bucket + key
modelo_servido = cargar_modelo_desde_lake(BUCKET, KEY_MODELO_V1)

# b) como lo haría el servicio: según el puntero de producción
modelo_prod, meta_prod = cargar_modelo_en_produccion()
print(f"En producción: {meta_prod['nombre']} {meta_prod['version']} | métricas {meta_prod['metricas']}")

pedido = pd.DataFrame([{
    "sucursal_id": 3, "sku_id": 42, "dia_semana": 5, "mes": 12, "es_finde": 1, "es_feriado": 0,
    "en_promocion": 1, "precio_unitario": 2100.0, "demanda_lag_1": 64.0, "demanda_lag_7": 58.0,
    "media_movil_7": 61.5,
}])[FEATURES]
print(f"Predicción (sucursal 3, SKU 42, sábado de diciembre, en promo): "
      f"{modelo_prod.predict(pedido)[0]:.1f} unidades")
assert np.allclose(modelo_servido.predict(X_test), modelo_prod.predict(X_test))


En producción: gout-demanda-rf v1 | métricas {'mae': 6.261, 'r2': 0.8933}
Predicción (sucursal 3, SKU 42, sábado de diciembre, en promo): 92.3 unidades


### 4.b Versión nueva sin romper la que está en producción

Entreno una v2 (más árboles y hojas más chicas), la subo a `models/v2/` y **todavía no la
promuevo**: el servicio sigue sirviendo v1 porque el puntero no cambió. Promover es una escritura
de `production.json`, y volver atrás es otra.


In [ ]:
HIPER_V2 = {"n_estimators": 250, "max_depth": 14, "min_samples_leaf": 5}
modelo_v2, metricas_v2, _ = entrenar(datos, **HIPER_V2)
try:
    subir_modelo(modelo_v2, "v2", metricas_v2, HIPER_V2, KEY_CURADO)
except FileExistsError as e:
    print("Aviso:", e)
print(f"v1 {metricas_v1}  |  v2 {metricas_v2}")

print("\nAntes de promover, el servicio sigue en:", cargar_modelo_en_produccion()[1]["version"])

# Reintentar subir v1 falla: las versiones publicadas son inmutables.
try:
    subir_modelo(modelo_v2, "v1", metricas_v2, HIPER_V2, KEY_CURADO)
except FileExistsError as e:
    print("Protección de versiones:", e)

if metricas_v2["mae"] < metricas_v1["mae"]:
    promover("v2")
print("Ahora el servicio carga:", cargar_modelo_en_produccion()[1]["version"])
promover("v1")                                    # rollback: una sola escritura
print("Tras el rollback carga:", cargar_modelo_en_produccion()[1]["version"])


v1 {'mae': 6.261, 'r2': 0.8933}  |  v2 {'mae': 5.833, 'r2': 0.9071}

Antes de promover, el servicio sigue en: v1
Protección de versiones: s3://datalake-tp6/models/v1/gout-demanda-rf.pkl ya existe: usá una versión nueva
models/production.json -> v2
Ahora el servicio carga: v2
models/production.json -> v1
Tras el rollback carga: v1


### 4.c Predicciones al lake (para el resolver de GraphQL)

El servicio deja las predicciones por lote en `curated/preds/…`, versionadas con el modelo que
las produjo, y un resolver estilo GraphQL las lee desde ahí.


In [ ]:
version_prod = cargar_modelo_en_produccion()[1]["version"]
KEY_PREDS = f"curated/preds/{NOMBRE_MODELO}/{version_prod}/predicciones.parquet"
tabla_pred = X_test.reset_index(names="id").assign(
    prediccion=modelo_prod.predict(X_test), version_modelo=version_prod)
s3.put_object(Bucket=BUCKET, Key=KEY_PREDS, Body=tabla_pred.to_parquet(index=False))


def resolver_prediccion(id_muestra, key=KEY_PREDS):
    t = leer_parquet(BUCKET, key)
    fila = t[t["id"] == id_muestra]
    return None if fila.empty else {"id": int(id_muestra),
                                    "prediccion": round(float(fila["prediccion"].iloc[0]), 2),
                                    "version_modelo": fila["version_modelo"].iloc[0]}


primer_id = int(tabla_pred["id"].iloc[0])
print(f"Guardadas {len(tabla_pred):,} predicciones en {KEY_PREDS}")
print(f"resolver_prediccion({primer_id}) ->", resolver_prediccion(primer_id))


Guardadas 1,600 predicciones en curated/preds/gout-demanda-rf/v1/predicciones.parquet
resolver_prediccion(2215) -> {'id': 2215, 'prediccion': 62.05, 'version_modelo': 'v1'}


## 5. (Opcional) MLflow apuntando a MinIO/S3


In [ ]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT      # MLflow habla con MinIO, no con AWS
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow_tp6.db")   # metadatos locales; artefactos al lake
cliente = MlflowClient()
NOMBRE_EXP = "mini-tp6-gout"
exp = cliente.get_experiment_by_name(NOMBRE_EXP)
exp_id = exp.experiment_id if exp else cliente.create_experiment(
    NOMBRE_EXP, artifact_location=f"s3://{BUCKET}/mlflow")
mlflow.set_experiment(experiment_id=exp_id)

modelo_v1, meta_v1 = cargar_modelo_en_produccion()   # el artefacto que se sirve, bajado del lake
with mlflow.start_run(run_name=f"{NOMBRE_MODELO}-{meta_v1['version']}") as run:
    mlflow.log_params(meta_v1["hiperparametros"])
    mlflow.log_metrics(meta_v1["metricas"])
    mlflow.set_tags({"version_lake": meta_v1["version"], "dataset": meta_v1["dataset"],
                     "sha256": meta_v1["sha256"]})
    joblib.dump(modelo_v1, f"{NOMBRE_MODELO}.pkl")
    mlflow.log_artifact(f"{NOMBRE_MODELO}.pkl", artifact_path="model")
    mlflow.log_dict(meta_v1, "model/metadata.json")
print("Run registrado:", run.info.run_id)
print(f"\nArtefactos de MLflow en s3://{BUCKET}/mlflow:")
listar(BUCKET, prefix="mlflow")


2026/10/05 18:26:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/05 18:26:58 INFO mlflow.store.db.utils: Updating database tables


Run registrado: c3df68ba63cc4bacb41fb62ef598542d

Artefactos de MLflow en s3://datalake-tp6/mlflow:
  mlflow/c3df68ba63cc4bacb41fb62ef598542d/artifacts/model/gout-demanda-rf.pkl  4,144,209 bytes
  mlflow/c3df68ba63cc4bacb41fb62ef598542d/artifacts/model/metadata.json        669 bytes


## 6. Reflexión

**1. ¿Por qué cargar el modelo desde el lake y no hornearlo en la imagen?**
Porque desacopla dos ciclos de vida que cambian a ritmos distintos: el código del servicio
cambia poco y el modelo se reentrena seguido. Con el modelo en el lake, publicar una versión
nueva es subir un objeto y mover un puntero, sin reconstruir ni redesplegar la imagen, que
además queda más liviana: acá el artefacto pesa 4 MB, pero en otros modelos son gigabytes. El
lake es el único origen de verdad: el endpoint REST, el servidor gRPC y el consumidor de
streaming cargan exactamente el mismo artefacto, con su `metadata.json` (dataset de origen,
hiperparámetros, métricas, checksum). El costo es que el servicio depende del lake para
arrancar. Lo mitigaría con reintentos, una caché local del último modelo válido y la
verificación del checksum que ya hace `cargar_modelo_en_produccion`. También conviene guardar en
la metadata la versión de scikit-learn, porque un pickle sólo se carga bien con la misma versión.

**2. ¿Cómo versionar un modelo nuevo sin romper el que está en producción?**
Lo hice en la sección 4.b. Cada versión vive en su propio prefijo inmutable
(`models/v2/…`) y subir sobre una versión existente da error. El servicio no apunta a una ruta
fija sino a `models/production.json`. La v2 (MAE 5,83 contra 6,26) se subió sin afectar a la
v1 en producción. Recién después de compararla se promovió con una sola escritura del puntero, y
el rollback fue otra escritura. En una plataforma real agregaría una etapa previa a la promoción
(*shadow* o *canary* sobre tráfico real), o usaría los alias del Model Registry de MLflow, que
implementan la misma idea.

**3. ¿Qué zona usar para las predicciones que sirve la API por GraphQL?**
`curated` (gold), con un prefijo por modelo y versión: `curated/preds/gout-demanda-rf/v1/`.
Las predicciones son un dato derivado, tipado y listo para consumir, que es justamente lo que
define a esa zona, y el resolver de GraphQL tiene que leer de un origen confiable. Versionarlas
por modelo permite saber qué versión generó cada predicción y comparar versiones. No van en
`models`, que es sólo para artefactos, ni en `raw`, que es para el dato como llega. Lo que sí iría
a `raw` son los pedidos que recibe la API (`raw/eventos/fecha=…`): sirven para auditar y, una
vez que se conoce la demanda real, para reentrenar.